# Averaged point metrics reward mode collapse

Every case of a loan application process starts with *Submit application*. With probability $p$ it takes
the fast track (*Automatic check*, *Approve*), otherwise the manual review (*Request documents*,
*Manual assessment*, *Final decision*). Each activity's time since the previous event is lognormal.

Because the true suffix distribution is known, we compare an **oracle** that samples from it with a
**mode model** that always predicts the most probable suffix, each activity at its median time. Both are
scored on every prefix of every simulated test case with the averaged point metrics of the current
framework: mean DLS (higher is better), and the MAE of suffix length and remaining time in days (lower is
better). A framework that rewards the true distribution must score the oracle at least as well.

In [ ]:
import numpy as np
import pandas as pd

from improper_suffix_metrics.evaluation import compare
from improper_suffix_metrics.markov_process import END, Activity, MarkovProcess
from improper_suffix_metrics.model import LOAN_APPLICATION, random_markov_process

SEED = 0
CASE_COUNT = 1000
DRAW_COUNT = 20
SETTING_COUNT = 500

## One setting

70% of the applications take the fast track, which takes hours, while the manual review takes about a
week.

In [ ]:
process = MarkovProcess(
    start={'Submit application': 1.0},
    activities={
        'Submit application': Activity(
            0.1, 0.5, {'Automatic check': 0.7, 'Request documents': 0.3}
        ),
        'Automatic check': Activity(1.0 / 24.0, 0.5, {'Approve': 1.0}),
        'Approve': Activity(0.5, 0.5, {END: 1.0}),
        'Request documents': Activity(2.0, 0.5, {'Manual assessment': 1.0}),
        'Manual assessment': Activity(4.0, 0.5, {'Final decision': 1.0}),
        'Final decision': Activity(1.0, 0.5, {END: 1.0}),
    },
)
pd.DataFrame(compare(process, CASE_COUNT, DRAW_COUNT, np.random.default_rng(SEED))).T.round(3)

## Random settings

Each setting draws $p \sim U(0, 1)$, the flat Dirichlet on two branches, and for each activity a median time log-uniform between one
hour and ten days and a log-scale spread $\sim U(0.1, 1)$. For each metric we report how often the
framework scores the mode model at least as well as the oracle, and the median relative advantage of
the mode model, positive when it scores better.

In [ ]:
def run(seed: np.random.SeedSequence) -> dict[str, dict[str, float]]:
    rng = np.random.default_rng(seed)
    return compare(random_markov_process(LOAN_APPLICATION, rng), CASE_COUNT, DRAW_COUNT, rng)


results = [run(seed) for seed in np.random.SeedSequence(SEED).spawn(SETTING_COUNT)]
oracle = pd.DataFrame([result['oracle'] for result in results])
mode = pd.DataFrame([result['mode'] for result in results])

sign = pd.Series({'dls': 1.0, 'length_mae': -1.0, 'remaining_time_mae': -1.0})
advantage = sign * (mode - oracle) / oracle
pd.DataFrame(
    {
        'mode at least as good': (advantage >= 0.0).mean(),
        'median relative advantage': advantage.median(),
    }
).round(3)